# LRCN Training Documentation

This notebook documents the current LRCN training workflow for the KTH action recognition project. It explains the training pipeline from dataset preparation to model evaluation and should be treated as the cleaned documentation for the project’s training stage.

The notebook follows the same workflow as the main training file, but it is organized in a more readable and explanatory manner for documentation purposes.

## 1. Objective

The goal of this project is to classify human actions from the KTH dataset using a Long-term Recurrent Convolutional Network (LRCN). The training pipeline prepares video clips, builds a dataset, trains a CNN + LSTM model, evaluates it on validation and test data, and then records the performance metrics.

In [ ]:
import os
import re
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt

## 2. Dataset path and class definitions

The KTH dataset is organized into action folders. Each class is mapped to a numeric label before building the training pipeline.

In [ ]:
base_dir = Path(r"C:\NNDL Lab Proj\KTH")
dataset_path = base_dir / "Dataset" if (base_dir / "Dataset").exists() else base_dir

classes = [
    "walking",
    "running",
    "handwaving",
    "handclapping"
]

print("Dataset path:", dataset_path)
print("Classes:", classes)

## 3. Collect video files and labels

Each video clip is stored inside its class directory. The code reads all .avi files and assigns the corresponding label index.

In [ ]:
video_paths = []
labels = []

for label, class_name in enumerate(classes):
    class_path = dataset_path / class_name

    for video_file in sorted(class_path.glob("*.avi")):
        video_paths.append(video_file)
        labels.append(label)

print("Total videos:", len(video_paths))
print("Total labels:", len(labels))
print("First 5 entries:")
for i in range(5):
    print(video_paths[i], "->", labels[i])

In [ ]:
class_counts = Counter(labels)

for class_id, class_name in enumerate(classes):
    print(f"{class_name}: {class_counts[class_id]}")

## 4. Person-based split

To avoid data leakage, videos are split by person ID rather than by random selection.

- training persons: 11 to 18
- validation persons: 19,20,21,23,24,25,1,4
- test persons: 22,2,3,5,6,7,8,9,10

In [ ]:
person_ids = []

for path in video_paths:
    match = re.search(r"person(\d+)", path.name)
    person_ids.append(int(match.group(1)))

unique_persons = sorted(set(person_ids))
print("Number of persons:", len(unique_persons))
print("Persons:", unique_persons)

train_persons = [11, 12, 13, 14, 15, 16, 17, 18]
val_persons = [19, 20, 21, 23, 24, 25, 1, 4]
test_persons = [22, 2, 3, 5, 6, 7, 8, 9, 10]

train_videos = []
train_labels = []
val_videos = []
val_labels = []
test_videos = []
test_labels = []

for video_path, label, person_id in zip(video_paths, labels, person_ids):
    if person_id in train_persons:
        train_videos.append(video_path)
        train_labels.append(label)
    elif person_id in val_persons:
        val_videos.append(video_path)
        val_labels.append(label)
    elif person_id in test_persons:
        test_videos.append(video_path)
        test_labels.append(label)

print("Training videos:", len(train_videos))
print("Validation videos:", len(val_videos))
print("Testing videos:", len(test_videos))

In [ ]:
print("TRAIN:", Counter(train_labels))
print("VALIDATION:", Counter(val_labels))
print("TEST:", Counter(test_labels))

## 5. Dataset class for video clips

Each video is sampled into a short temporal sequence of 16 frames with a stride of 6. Each frame is converted to RGB, resized to 112x112, normalized, and transformed from HWC to CHW format for PyTorch processing.

In [ ]:
class KTHDataset(Dataset):

    def __init__(self, video_paths, labels, num_frames=16, frame_step=6, image_size=112):
        self.video_paths = video_paths
        self.labels = labels
        self.num_frames = num_frames
        self.frame_step = frame_step
        self.image_size = image_size

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, index):
        video_path = self.video_paths[index]
        label = self.labels[index]

        cap = cv2.VideoCapture(str(video_path))
        frames = []

        sample_indices = np.arange(0, self.num_frames * self.frame_step, self.frame_step)

        for frame_index in sample_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
            ret, frame = cap.read()

            if not ret:
                cap.release()
                raise RuntimeError(f"Could not read frame {frame_index} from {video_path}")

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (self.image_size, self.image_size))
            frame = frame.astype(np.float32) / 255.0
            frame = torch.from_numpy(frame).permute(2, 0, 1)
            frames.append(frame)

        cap.release()
        video = torch.stack(frames)
        return video, torch.tensor(label, dtype=torch.long)

In [ ]:
train_dataset = KTHDataset(train_videos, train_labels)
video, label = train_dataset[0]

print("Video shape:", video.shape)
print("Label:", label)
print("Video dtype:", video.dtype)
print("Min:", video.min().item())
print("Max:", video.max().item())

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
videos, labels = next(iter(train_loader))

print("Batch video shape:", videos.shape)
print("Batch label shape:", labels.shape)
print("Labels:", labels)

## 6. LRCN model architecture

The model combines a CNN for spatial feature extraction and an LSTM for temporal learning. This is a standard LRCN design: each frame is processed individually by the CNN, then the sequence of features is passed to the LSTM.

In [ ]:
class CNNFeatureExtractor(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return x


class LRCN(nn.Module):

    def __init__(self, num_classes=4, dropout=0.3):
        super().__init__()

        self.cnn = CNNFeatureExtractor()

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=128,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        B, T, C, H, W = x.shape

        x = x.view(B * T, C, H, W)
        x = self.cnn(x)
        x = x.view(B, T, 128)

        output, (hidden, cell) = self.lstm(x)
        x = hidden[-1]
        x = self.dropout(x)
        x = self.fc(x)

        return x

In [ ]:
model = LRCN(num_classes=4)
print(model)

# Count parameters
print("Total parameters:", sum(p.numel() for p in model.parameters()))
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
prediction = model(videos)
print("Input shape:", videos.shape)
print("Output shape:", prediction.shape)
print("Prediction:", prediction)

## 7. Training setup

The model is trained using the cross-entropy loss and Adam optimizer. Validation data is used to monitor performance and select the best model.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = LRCN(num_classes=4).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    verbose=True
)

val_dataset = KTHDataset(val_videos, val_labels)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

## 8. Training loop

The model is trained for multiple epochs, and after each epoch it records training and validation loss along with accuracy. The best-performing checkpoint is saved for evaluation.

In [ ]:
num_epochs = 40

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

best_val_accuracy = 0.0
best_epoch = 0
checkpoint_path = base_dir / "best_lrcn_model.pth"

print(f"Starting training for {num_epochs} epochs...")
print(f"Checkpoint path: {checkpoint_path}")

for epoch in range(num_epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for videos_batch, labels_batch in train_loader:
        videos_batch = videos_batch.to(device)
        labels_batch = labels_batch.to(device)

        predictions = model(videos_batch)
        loss = criterion(predictions, labels_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        predicted = torch.argmax(predictions, dim=1)
        total += labels_batch.size(0)
        correct += (predicted == labels_batch).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    model.eval()
    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for val_videos_batch, val_labels_batch in val_loader:
            val_videos_batch = val_videos_batch.to(device)
            val_labels_batch = val_labels_batch.to(device)

            predictions = model(val_videos_batch)
            loss = criterion(predictions, val_labels_batch)

            running_val_loss += loss.item()
            predicted = torch.argmax(predictions, dim=1)

            val_total += val_labels_batch.size(0)
            val_correct += (predicted == val_labels_batch).sum().item()

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = 100 * val_correct / val_total
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_epoch = epoch + 1
        torch.save(model.state_dict(), checkpoint_path)
        marker = " ** BEST **"
    else:
        marker = ""

    current_lr = optimizer.param_groups[0]['lr']
    print(
        f"Epoch [{epoch+1:2d}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_accuracy:.2f}% "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_accuracy:.2f}% "
        f"LR: {current_lr:.6f}{marker}"
    )

print("Training complete.")
print(f"Best validation accuracy: {best_val_accuracy:.2f}% at epoch {best_epoch}")
print(f"Model saved to: {checkpoint_path}")

## 9. Training curves

The training and validation curves are used to understand whether the model is learning and whether it is overfitting.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, num_epochs + 1)

ax1.plot(epochs_range, train_losses, 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs_range, val_losses, 'r-', label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training vs Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, train_accuracies, 'b-', label='Train Accuracy', linewidth=2)
ax2.plot(epochs_range, val_accuracies, 'r-', label='Val Accuracy', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training vs Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(base_dir / "training_curves.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved training curves to: {base_dir / 'training_curves.png'}")

## 10. Test evaluation

After training, the best model is loaded and evaluated on the unseen test set. Metrics include overall accuracy, per-class accuracy, confusion matrix, and probability output for sample videos.

In [ ]:
model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
model.eval()

test_dataset = KTHDataset(test_videos, test_labels)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)

test_correct = 0
test_total = 0
all_preds = []
all_true = []

with torch.no_grad():
    for videos_batch, labels_batch in test_loader:
        videos_batch = videos_batch.to(device)
        labels_batch = labels_batch.to(device)

        predictions = model(videos_batch)
        predicted = torch.argmax(predictions, dim=1)

        test_total += labels_batch.size(0)
        test_correct += (predicted == labels_batch).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_true.extend(labels_batch.cpu().numpy())

test_accuracy = 100 * test_correct / test_total
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Test Correct: {test_correct} / {test_total}")

print("\nPer-class accuracy:")
for class_id, class_name in enumerate(classes):
    class_mask = [i for i, t in enumerate(all_true) if t == class_id]
    class_correct = sum(1 for i in class_mask if all_preds[i] == class_id)
    class_acc = 100 * class_correct / len(class_mask) if class_mask else 0
    print(f"  {class_name}: {class_acc:.2f}% ({class_correct}/{len(class_mask)})")

print("\nConfusion Matrix:")
print(f"{'':>15}", end="")
for name in classes:
    print(f"{name:>14}", end="")
print()

for true_id, true_name in enumerate(classes):
    print(f"{true_name:>15}", end="")
    for pred_id in range(len(classes)):
        count = sum(1 for t, p in zip(all_true, all_preds) if t == true_id and p == pred_id)
        print(f"{count:>14}", end="")
    print()

print(f"\nPrediction distribution: {Counter(all_preds)}")
print(f"True label distribution: {Counter(all_true)}")

In [ ]:
model.eval()

sample_indices = []
for class_id in range(4):
    class_idxs = [i for i, lb in enumerate(test_labels) if lb == class_id]
    sample_indices.extend(class_idxs[:3])
sample_indices = sample_indices[:10]

with torch.no_grad():
    for idx in sample_indices:
        video = test_dataset[idx][0].unsqueeze(0).to(device)
        true_label = test_labels[idx]

        logits = model(video)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_label = np.argmax(probs)

        status = "OK" if pred_label == true_label else "WRONG"

        print(f"\n{status} Video: {test_videos[idx].name}")
        print(f"  Actual:    {classes[true_label]} (class {true_label})")
        print(f"  Predicted: {classes[pred_label]} (class {pred_label})")
        print("  Probabilities:")
        for ci, cn in enumerate(classes):
            bar = "#" * int(probs[ci] * 30)
            print(f"    {cn:>14}: {probs[ci]:.4f} {bar}")

## Summary

This notebook documents the full training workflow for the KTH LRCN project. It shows how raw videos are transformed into in-model input, how they are split for training and evaluation, how the temporal model is defined, and how the final performance is measured.

It is the documentation version of the current training stage and acts as the formal record of the model-building and evaluation pipeline.